# Lecture 8: Vector Stores ” Qdrant & Pinecone

**Course:** NLP with LangChain | **Platform:** Hope to Skill  
**Duration:** ~20 minutes | **Level:** Intermediate  

---

## The Big Picture

In Lecture 5 we **loaded** documents. In Lecture 6 we **split** them into chunks.  
In Lecture 7 we **embedded** them into numbers. But where do we *store* millions  
of these number-lists and search through them in milliseconds?

> Regular databases (PostgreSQL, MySQL) are like a **library with no catalog** â€”  
> you'd have to read the summary of every single book to find what you need.  
> Vector databases are like a **smart librarian** who instantly knows  
> which shelf has what you're looking for.

### What You Will Learn

| # | Topic | Real-World Analogy |
|---|-------|--------------------|
| 1 | Why regular databases fail for vectors | Library without a catalog |
| 2 | What is a vector database? | A smart librarian for your embeddings |
| 3 | Qdrant â€” open-source power | Your own private library |
| 4 | Pinecone â€” managed simplicity | A library-as-a-service |
| 5 | Core operations (upsert, query, filter, delete) | Add books, find books, organize shelves |
| 6 | HNSW algorithm (simple explanation) | Express elevator vs stairs |
| 7 | Hands-on: build a searchable movie database | Build it yourself! |

> **After this lecture:** You'll be able to store embeddings in a real vector  
> database and search through them by meaning â€” the core of every RAG system!

---

## 0. Environment Setup

Run this cell **once** to install the packages we'll need today.
We'll use **Qdrant Cloud** â€” the free tier gives you 1GB of storage!

In [1]:
# Install required packages (run once, then you can skip this cell)
# qdrant-client: Python client for Qdrant vector database
# sentence-transformers: free embedding models (same as Lecture 7)
# langchain-text-splitters: text splitting utilities
%pip install qdrant-client sentence-transformers langchain langchain-community langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


---

## 1. Why Can't We Use Regular Databases?

### The Problem

PostgreSQL, MySQL, and MongoDB are **amazing** for structured data:  
finding a user by ID, filtering products by price, sorting by date.

But embeddings are **lists of 384+ numbers**. To find "similar" vectors,  
a regular database would have to:

1. Load **every single vector** from storage
2. Calculate the similarity between your query and **each one**
3. Sort all results and return the top 5



> **Analogy:** Imagine searching for a book by its *topic* in a library  
> with no catalog. You'd have to read the summary of **every book**.  
> A vector database builds a smart index â€” like a topic catalog â€”  
> so you can jump straight to the right shelf.


---

## 2. What Is a Vector Database?

A vector database is a **specialized database built for storing and  
searching embeddings** (lists of numbers that represent meaning).

### Core Operations

| Operation | What It Does | Real-World Analogy |
|-----------|-------------|-------------------|
| **Upsert** | Add or update vectors + metadata | Adding a book to the library shelf |
| **Query** | Find the K most similar vectors | Asking the librarian for recommendations |
| **Filter** | Narrow search by metadata | "Only show me sci-fi books from 2024" |
| **Delete** | Remove vectors by ID or filter | Removing a book from the collection |

### What Makes It Special?

| Regular Database | Vector Database |
|-----------------|----------------|
| `SELECT * FROM docs WHERE title = 'AI'` | "Find me documents *similar to this meaning*..." |
| Exact match only | Meaning match! |
| Great for: IDs, names, prices | Great for: search by concept, Q&A, recommendations |

The core question a vector database answers:

> **"Give me the 5 vectors most similar to this query vector"**  
> (optionally filtered by metadata like date, category, or source)

---

## 3. Qdrant: Open-Source Power

**Qdrant** (pronounced "quadrant") is an open-source vector database  
written in **Rust** â€” one of the fastest programming languages.

### Why Choose Qdrant?

| Feature | Details |
|---------|---------|
| **Language** | Written in Rust â€” extremely fast |
| **Deployment** | Self-hosted (free) OR Qdrant Cloud |
| **Free Tier** | 4 GB free on Qdrant Cloud |
| **Filtering** | Advanced payload filtering built-in |
| **Hybrid Search** | Combines dense + sparse vectors |
| **Community** | Active open-source community |

### Three Ways to Use Qdrant

| Mode | Setup | Best For |
|------|-------|----------|
| **In-Memory** | `QdrantClient(":memory:")` | Quick testing (data disappears on restart) |
| **Local File** | `QdrantClient(path="./qdrant_data")` | Small projects, prototyping |
| **Cloud/Server** | `QdrantClient(url="...", api_key="...")` | Learning & production (what we'll use!) |

> **For this lecture:** We'll use **Qdrant Cloud** â€” the free tier gives you 1GB!
> Your data persists in the cloud, and you can view your collections in the
> Qdrant dashboard â€” perfect for learning how vector stores actually work.

---

## 4. Pinecone: Managed Simplicity

**Pinecone** is a fully managed vector database â€” you don't run any  
servers or infrastructure. Just call the API.

### Why Choose Pinecone?

| Feature | Details |
|---------|---------|
| **Setup** | Zero infrastructure â€” fully managed cloud |
| **Scaling** | Auto-scales based on usage |
| **Uptime** | 99.9% uptime SLA |
| **API** | Simple REST API + Python SDK |
| **Free Tier** | 1 index, 100K vectors free |

### Pricing Overview

| Tier | Cost | Includes |
|------|------|----------|
| Free (Starter) | $0/month | 1 index, 100K vectors, 2GB storage |
| Standard | ~$70/month | Multiple indexes, more storage |
| Enterprise | Custom | SLA, dedicated support, compliance |

> **Note:** We won't use Pinecone in the hands-on because it requires  
> creating an account and API key. But we'll show the code pattern  
> so you can switch to Pinecone anytime â€” the concepts are identical!

---

## 5. Qdrant vs Pinecone Side by Side

| Feature | Qdrant | Pinecone |
|---------|--------|----------|
| **Cost** | Free (self-hosted) / Free tier (cloud) | Free tier then $70+/month |
| **Setup Time** | ~5 min (cloud) / seconds (in-memory) | ~2 min (cloud only) |
| **Open Source** | Yes (Apache 2.0) | No (proprietary) |
| **Self-Hosting** | Yes â€” Docker, Kubernetes | No â€” cloud only |
| **Customization** | High â€” full control over config | Limited â€” managed service |
| **Hybrid Search** | Built-in (dense + sparse) | Available |
| **Scaling** | Manual (self-hosted) / Auto (cloud) | Fully automatic |
| **Best For** | Full control, cost-conscious teams | Fast deployment, small teams |

### Our Recommendation for Beginners

```
Starting a new project?
  |-- Want free + full control?       --> Qdrant (cloud free tier or self-hosted)
  |-- Want zero setup + small cost?    --> Pinecone (free tier)
  |-- Building production RAG?         --> Either works! Choose based on team skills
```

> **For this course:** We use **Qdrant Cloud** (free tier) because it's free,
> easy to set up, and lets you see your data in the dashboard.
> The code patterns transfer directly to Pinecone or any other vector DB.

---

## 6. Core Operations  Upsert, Query, Filter, Delete

Let's learn the four main operations of a vector database by building  
a **movie recommendation search engine**!

We'll store 10 movies with their descriptions as embeddings and then  
search for movies by meaning â€” not just by title or genre.

### Our Movie Database Plan

```
Step 1: Create a collection (like creating a table)
Step 2: Embed movie descriptions (convert text to numbers)
Step 3: Upsert movies into Qdrant (store the vectors + metadata)
Step 4: Query by meaning ("find movies about space travel")
Step 5: Filter by metadata ("only sci-fi movies")
Step 6: Delete movies from the collection
```

In [ ]:
# Step 1: Set up Qdrant Cloud and create a collection

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from sentence_transformers import SentenceTransformer

# Load our embedding model (same as Lecture 7)
model = SentenceTransformer("all-MiniLM-L6-v2")

# ============================================================
# QDRANT CLOUD SETUP (Free Tier â€” 1GB Storage)
# ============================================================
# 1. Sign up at: https://cloud.qdrant.io/
# 2. Create a cluster (free tier available)
# 3. Copy your cluster URL and API key from the dashboard
# 4. Replace the placeholders below with your credentials
# ============================================================

QDRANT_URL = ""        # e.g., "https://69b88521-b097-492f-971c-9cf0a9s-west-2-0.aws.cloud.qdrant.io:6333"
QDRANT_API_KEY = ""

# Connect to Qdrant Cloud
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

# Create a "collection" â€” similar to a table in a regular database
# A collection stores vectors of a specific size with a specific distance metric
client.create_collection(
    collection_name="movies",
    vectors_config=VectorParams(
        size=384,                # Must match our model's output (384 for MiniLM)
        distance=Distance.COSINE,  # Cosine similarity (our default from Lecture 7)
    ),
)

print("Collection 'movies' created!")
print(f"  Vector size: 384 dimensions")
print(f"  Distance metric: Cosine Similarity")

# Check collection info
# .get_collection() returns details about the collection
info = client.get_collection("movies")
print(f"  Points count: {info.points_count} (empty â€” we haven't added anything yet)")

c:\Users\mhari\miniconda3\envs\demo1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 318.98it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection 'movies' created!
  Vector size: 384 dimensions
  Distance metric: Cosine Similarity
  Points count: 0 (empty â€” we haven't added anything yet)


#### What just happened?

- We connected to **Qdrant Cloud** using your cluster URL and API key
- We created a **collection** called `"movies"` â€” think of it as a table
- We configured it for **384-dimensional vectors** (matching our MiniLM model)
- We chose **cosine similarity** as the distance metric (same as Lecture 7)

**Key vocabulary:**
- **Collection** = a group of vectors (like a table in SQL)
- **Point** = a single vector + its metadata (like a row in SQL)
- **Payload** = metadata attached to a vector (like extra columns in SQL)

**Pro tip:** Visit your Qdrant Cloud dashboard to see the "movies" collection appear!

In [3]:
# Step 2: Prepare our movie data and generate embeddings

# 10 movies with title, year, genre, and description
# The DESCRIPTION is what we'll embed â€” it captures the movie's meaning
movies = [
    {
        "id": 1,
        "title": "The Matrix",
        "year": 1999,
        "genre": "sci-fi",
        "description": "A computer hacker discovers that reality is a simulation created by machines to enslave humanity.",
    },
    {
        "id": 2,
        "title": "Inception",
        "year": 2010,
        "genre": "sci-fi",
        "description": "A skilled thief who steals corporate secrets through dream-sharing technology is given a chance to erase his criminal record.",
    },
    {
        "id": 3,
        "title": "The Godfather",
        "year": 1972,
        "genre": "crime",
        "description": "The aging patriarch of an organized crime dynasty transfers control of his empire to his reluctant youngest son.",
    },
    {
        "id": 4,
        "title": "Pulp Fiction",
        "year": 1994,
        "genre": "crime",
        "description": "Interconnected stories of criminals, a boxer, and a pair of bandits unfold in non-linear fashion in Los Angeles.",
    },
    {
        "id": 5,
        "title": "The Shawshank Redemption",
        "year": 1994,
        "genre": "drama",
        "description": "A banker sentenced to life in prison forms a transformative friendship with a fellow inmate over two decades.",
    },
    {
        "id": 6,
        "title": "Forrest Gump",
        "year": 1994,
        "genre": "drama",
        "description": "A man with a low IQ but good intentions witnesses and unwittingly influences several major historical events in America.",
    },
    {
        "id": 7,
        "title": "The Dark Knight",
        "year": 2008,
        "genre": "action",
        "description": "Batman faces the Joker, a criminal mastermind who wants to plunge Gotham City into anarchy and chaos.",
    },
    {
        "id": 8,
        "title": "Interstellar",
        "year": 2014,
        "genre": "sci-fi",
        "description": "A team of astronauts travels through a wormhole near Saturn in search of a new habitable planet for humanity.",
    },
    {
        "id": 9,
        "title": "Titanic",
        "year": 1997,
        "genre": "romance",
        "description": "A young couple from different social classes fall in love aboard the ill-fated maiden voyage of the RMS Titanic.",
    },
    {
        "id": 10,
        "title": "The Lion King",
        "year": 1994,
        "genre": "animation",
        "description": "A young lion prince flees his kingdom after the murder of his father and learns the true meaning of responsibility and bravery.",
    },
]

# Extract just the descriptions for embedding
# List comprehension: creates a new list pulling only "description" from each movie
descriptions = [movie["description"] for movie in movies]

# Embed all 10 descriptions at once (faster than one at a time)
movie_embeddings = model.encode(descriptions)

print(f"Prepared {len(movies)} movies")
print(f"Generated {movie_embeddings.shape[0]} embeddings, each with {movie_embeddings.shape[1]} dimensions")

# Preview: show each movie title and the first 3 numbers of its embedding
# enumerate() gives us the index (i=0,1,2...) and the movie dict
for i, movie in enumerate(movies):
    # movie_embeddings[i] gets the embedding for movie at index i
    # [:3] shows only the first 3 of 384 numbers
    # :<25 left-aligns the title in a 25-character wide column
    print(f"  {movie['title']:<25} -> [{movie_embeddings[i][0]:.3f}, {movie_embeddings[i][1]:.3f}, {movie_embeddings[i][2]:.3f}, ...]")

Prepared 10 movies
Generated 10 embeddings, each with 384 dimensions
  The Matrix                -> [-0.065, 0.049, -0.100, ...]
  Inception                 -> [-0.110, 0.077, 0.015, ...]
  The Godfather             -> [0.004, 0.007, -0.046, ...]
  Pulp Fiction              -> [0.023, -0.023, -0.070, ...]
  The Shawshank Redemption  -> [-0.026, 0.058, -0.103, ...]
  Forrest Gump              -> [0.052, -0.011, -0.054, ...]
  The Dark Knight           -> [0.012, 0.027, -0.067, ...]
  Interstellar              -> [0.017, -0.067, 0.014, ...]
  Titanic                   -> [0.003, 0.005, 0.062, ...]
  The Lion King             -> [0.021, 0.077, -0.035, ...]


#### What just happened?

- We created a list of 10 movies, each with `title`, `year`, `genre`, and `description`
- We extracted the descriptions and embedded them into 384-dimensional vectors
- Each movie's meaning is now captured as a list of 384 numbers

**Why embed the description?** Because that's what captures the movie's  
*meaning*. The title alone ("The Matrix") doesn't tell the model much,  
but the description ("computer hacker discovers reality is a simulation...")  
gives it rich semantic information to work with.

In [4]:
# Step 3: UPSERT â€” Add all movies to Qdrant

from qdrant_client.models import PointStruct

# Build a list of "points" â€” each point is a vector + metadata
# A point = one row in our vector database
points = []

# This loop creates a PointStruct for each movie
# enumerate() gives us the index (i) and the movie dict
for i, movie in enumerate(movies):
    point = PointStruct(
        id=movie["id"],                          # Unique ID for this point
        vector=movie_embeddings[i].tolist(),      # The 384-dim embedding (.tolist() converts numpy to plain list)
        payload={                                 # Metadata â€” searchable and filterable
            "title": movie["title"],
            "year": movie["year"],
            "genre": movie["genre"],
            "description": movie["description"],
        },
    )
    points.append(point)

# Upsert = "update or insert"
# If the ID exists â†’ update it; if not â†’ insert it
# This is safer than plain insert because it won't fail on duplicate IDs
client.upsert(
    collection_name="movies",
    points=points,
)

# ============================================================
# CREATE PAYLOAD INDEXES for filtering (required for Qdrant Cloud!)
# ============================================================
# Qdrant Cloud requires indexes on payload fields before you can filter by them
# We'll create indexes for "genre" and "year" since we filter by these fields

client.create_payload_index(
    collection_name="movies",
    field_name="genre",
    field_schema="keyword",  # "keyword" for exact text matching
)

client.create_payload_index(
    collection_name="movies",
    field_name="year",
    field_schema="integer",  # "integer" for numeric fields
)

# Verify the upsert worked
info = client.get_collection("movies")
print(f"Successfully upserted {info.points_count} movies into Qdrant!")
print(f"\nEach movie is stored as:")
print(f"  - ID: unique identifier (1-10)")
print(f"  - Vector: 384 numbers (the embedding)")
print(f"  - Payload: title, year, genre, description (metadata)")
print(f"\nPayload indexes created for filtering:")
print(f"  - genre (keyword) â€” for exact genre matching")
print(f"  - year (integer) â€” for year range filtering")

Successfully upserted 10 movies into Qdrant!

Each movie is stored as:
  - ID: unique identifier (1-10)
  - Vector: 384 numbers (the embedding)
  - Payload: title, year, genre, description (metadata)

Payload indexes created for filtering:
  - genre (keyword) â€” for exact genre matching
  - year (integer) â€” for year range filtering


#### What just happened?

- We created **PointStruct** objects â€” each one bundles:
  - `id`: unique identifier (1-10)
  - `vector`: the 384-dimensional embedding (as a plain Python list)
  - `payload`: metadata dict (title, year, genre, description)

- We used **upsert** (not insert) â€” this is safer because:
  - If the ID doesn't exist â†’ it **inserts** a new point
  - If the ID already exists â†’ it **updates** the existing point
  - No "duplicate key" errors!

- We created **payload indexes** for "genre" and "year":
  - **Required for Qdrant Cloud** â€” you can't filter without indexes!
  - `field_schema="keyword"` for exact text matching (genre)
  - `field_schema="integer"` for numeric fields (year)
  - In-memory mode doesn't need indexes, but cloud/server mode does

- `.tolist()` converts the numpy array to a plain Python list (Qdrant needs this)

**Tip:** In production, upsert your data in **batches** (e.g., 100-500 points  
at a time) for better performance. Our 10 movies fit in a single batch.

In [5]:
# Step 4: QUERY â€” Similarity search (the magic!)

# Let's search for movies by MEANING, not keywords
query_text = "a movie about space travel and exploring other planets"

# Embed the search query using the SAME model we used for movies
# (This is critical â€” different models produce incompatible vectors!)
query_embedding = model.encode(query_text)

# Search Qdrant for the most similar vectors
# We use query_points() with manual vectors for Qdrant Cloud
results = client.query_points(
    collection_name="movies",
    query=query_embedding.tolist(),  # Must be a plain list, not numpy
    limit=3,                          # Return top 3 matches
).points

print(f"Query: '{query_text}'")
print(f"{'=' * 65}")
print(f"Top 3 most similar movies:")
print(results)

# results is a list of ScoredPoint objects
# Each result has: .id (point ID), .score (similarity), .payload (metadata)
# enumerate() gives us the rank (0, 1, 2) and the result object
for rank, result in enumerate(results):
    print(f"\n  #{rank + 1} â€” {result.payload['title']} ({result.payload['year']})")
    print(f"       Genre: {result.payload['genre']}")
    # :.4f shows 4 decimal places for the similarity score
    print(f"       Score: {result.score:.4f}")
    # [:80] shows first 80 characters of description
    # To see the full description, remove [:80]
    print(f"       Description: {result.payload['description'][:80]}...")

Query: 'a movie about space travel and exploring other planets'
Top 3 most similar movies:
[ScoredPoint(id=8, version=1, score=0.53049713, payload={'title': 'Interstellar', 'year': 2014, 'genre': 'sci-fi', 'description': 'A team of astronauts travels through a wormhole near Saturn in search of a new habitable planet for humanity.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=9, version=1, score=0.22281457, payload={'title': 'Titanic', 'year': 1997, 'genre': 'romance', 'description': 'A young couple from different social classes fall in love aboard the ill-fated maiden voyage of the RMS Titanic.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=7, version=1, score=0.12940481, payload={'title': 'The Dark Knight', 'year': 2008, 'genre': 'action', 'description': 'Batman faces the Joker, a criminal mastermind who wants to plunge Gotham City into anarchy and chaos.'}, vector=None, shard_key=None, order_value=None)]

  #1 â€” Interstellar (2014)
       Genr

#### What just happened?

We just performed **semantic search** â€” finding movies by *meaning*:

1. We embedded the query `"a movie about space travel and exploring other planets"`
2. Qdrant compared this query vector against all 10 movie vectors
3. It returned the 3 most similar movies, ranked by cosine similarity score

**Notice:** The query says "space travel" and "other planets" â€” neither phrase  
appears exactly in any movie description! But Qdrant found **Interstellar**  
("astronauts travel through a wormhole... new habitable planet")  
because the *meanings* are similar.

This is the same principle from Lecture 7 â€” but now the vectors are stored  
in a database that can scale to millions of documents!

In [6]:
# Step 5: FILTER â€” Narrow search by metadata

from qdrant_client.models import Filter, FieldCondition, MatchValue

# Search for "exciting action-packed battles" but ONLY in sci-fi movies
query_text = "exciting action-packed battles and conflict"
query_embedding = model.encode(query_text)

# --- Without filter: search ALL movies ---
results_all = client.query_points(
    collection_name="movies",
    query=query_embedding.tolist(),
    limit=3,
).points

# --- With filter: search ONLY sci-fi movies ---
results_filtered = client.query_points(
    collection_name="movies",
    query=query_embedding.tolist(),
    query_filter=Filter(
        must=[  # "must" means ALL conditions must be true (like AND in SQL)
            FieldCondition(
                key="genre",                      # Filter on the "genre" payload field
                match=MatchValue(value="sci-fi"),  # Only keep points where genre == "sci-fi"
            )
        ]
    ),
    limit=3,
).points

# Compare results side by side
print("WITHOUT FILTER (all genres):")
# This loop shows results from searching ALL movies
for rank, result in enumerate(results_all):
    print(f"  #{rank + 1} {result.payload['title']:<25} [{result.payload['genre']:<10}] score: {result.score:.4f}")

print(f"\nWITH FILTER (sci-fi only):")
# This loop shows results filtered to sci-fi genre only
for rank, result in enumerate(results_filtered):
    print(f"  #{rank + 1} {result.payload['title']:<25} [{result.payload['genre']:<10}] score: {result.score:.4f}")

print(f"\nThe filter narrowed results to only sci-fi movies!")    

WITHOUT FILTER (all genres):
  #1 Pulp Fiction              [crime     ] score: 0.3151
  #2 The Lion King             [animation ] score: 0.1518
  #3 The Matrix                [sci-fi    ] score: 0.1514

WITH FILTER (sci-fi only):
  #1 The Matrix                [sci-fi    ] score: 0.1514
  #2 Interstellar              [sci-fi    ] score: 0.0880
  #3 Inception                 [sci-fi    ] score: 0.0047

The filter narrowed results to only sci-fi movies!


#### What just happened?

- **Without filter:** Qdrant searched ALL 10 movies and returned the best  
  matches regardless of genre
- **With filter:** Qdrant first narrowed down to only sci-fi movies,  
  then searched within that subset

**Filter types available in Qdrant:**

| Filter | What It Does | Example |
|--------|-------------|---------|  
| `MatchValue` | Exact match on a field | `genre == "sci-fi"` |
| `Range` | Numeric range | `year >= 2000 AND year <= 2020` |
| `must` | ALL conditions required (AND) | genre = sci-fi AND year > 2000 |
| `should` | ANY condition works (OR) | genre = sci-fi OR genre = action |
| `must_not` | Exclude matches (NOT) | genre != romance |

Filters are applied **before** the similarity search â€” so the search  
only runs against vectors that match your filter criteria. This makes  
filtered searches even faster!

In [7]:
# Step 6: DELETE â€” Remove movies from the collection

from qdrant_client.models import PointIdsList

# Check count before deletion
info_before = client.get_collection("movies")
print(f"Before deletion: {info_before.points_count} movies")

# Delete movies by their IDs
# Let's remove Titanic (id=9) and The Lion King (id=10)
client.delete(
    collection_name="movies",
    points_selector=PointIdsList(
        points=[9, 10],  # List of point IDs to delete
    ),
)

# Check count after deletion
info_after = client.get_collection("movies")
print(f"After deletion:  {info_after.points_count} movies")
print(f"Removed: {info_before.points_count - info_after.points_count} movies (Titanic & The Lion King)")

# Verify: search for a romance movie â€” Titanic should be GONE
query_embedding = model.encode("romantic love story on a ship")
results = client.query_points(
    collection_name="movies",
    query=query_embedding.tolist(),
    limit=3,
).points

print(f"\nSearch for 'romantic love story on a ship':")
for rank, result in enumerate(results):
    print(f"  #{rank + 1} {result.payload['title']} â€” score: {result.score:.4f}")

print(f"\nTitanic is gone â€” it was deleted from the collection!")

Before deletion: 10 movies
After deletion:  8 movies
Removed: 2 movies (Titanic & The Lion King)

Search for 'romantic love story on a ship':
  #1 Pulp Fiction â€” score: 0.1598
  #2 The Shawshank Redemption â€” score: 0.1590
  #3 Interstellar â€” score: 0.1254

Titanic is gone â€” it was deleted from the collection!


#### What just happened?

- We deleted 2 movies (Titanic and The Lion King) by their IDs
- The collection went from 10 points to 8 points
- When we searched for "romantic love story on a ship", Titanic no longer appeared
- Deletion is permanent â€” the vectors and their metadata are removed

**Other ways to delete:**
- By **filter** (e.g., remove all movies before 1990): uses `FilterSelector`
- By **payload field** (e.g., remove all "romance" genre): also uses `FilterSelector`

Check the Qdrant docs for more deletion options.

---

## 8. Hands-On: Full RAG Pipeline (Lectures 5-8 Combined!)

Let's bring everything together! We'll build the complete pipeline:

```
Load (L5) --> Split (L6) --> Embed (L7) --> Store (L8) --> Search (L8)
```

We'll use the `nlp_article.txt` from Lecture 6 as our document.

In [ ]:
# ============================================================
# FULL RAG PIPELINE: Load -> Split -> Embed -> Store -> Search
# ============================================================
# EMBEDDING METHOD: SentenceTransformer (FREE, Local)
#
# This cell uses FREE local embeddings (no API key required)
# For OpenAI embeddings, see the next cell instead
#
# Pipeline Steps:
#   0. Setup (Model + Database)
#   1. Load (Read documents from disk)
#   2. Split (Break into chunks)
#   3. Embed (Convert to vectors with SentenceTransformer)
#   4. Store (Save in vector database)
# ============================================================

# ============================================================
# IMPORTS
# ============================================================
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer


# ============================================================
# STEP 0: SETUP - Initialize Model & Database Connection
# ============================================================
print("=" * 70)
print("STEP 0: SETUP - Initialize Model & Database Connection")
print("=" * 70)

# Load the SentenceTransformer model (FREE, runs locally)
model = SentenceTransformer("all-MiniLM-L6-v2")
EMBEDDING_DIM = 384  # SentenceTransformer output size
print("[SUCCESS] Model loaded: all-MiniLM-L6-v2 (384 dimensions)")

# Connect to Qdrant Cloud
QDRANT_URL = ""
QDRANT_API_KEY = ""
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
print("[SUCCESS] Connected to Qdrant Cloud")


# ============================================================
# STEP 1: LOAD - Read Documents from Disk
# ============================================================
print("\n" + "=" * 70)
print("STEP 1: LOAD - Read Documents from Disk")
print("=" * 70)

loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
raw_documents = loader.load()

print(f"[SUCCESS] Loaded: {len(raw_documents)} document(s)")
print(f"Preview: {raw_documents[0].page_content[:100]}...")


# ============================================================
# STEP 2: SPLIT - Break Documents into Chunks
# ============================================================
print("\n" + "=" * 70)
print("STEP 2: SPLIT - Break Documents into Chunks")
print("=" * 70)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = splitter.split_documents(raw_documents)

print(f"[SUCCESS] Split into: {len(chunks)} chunks")
print(f"Chunk size: 500 characters")
print(f"Chunk overlap: 50 characters")
print(f"First chunk preview: {chunks[0].page_content[:80]}...")


# ============================================================
# STEP 3: EMBED - Convert Text to Vectors (SentenceTransformer)
# ============================================================
print("\n" + "=" * 70)
print("STEP 3: EMBED - Convert Text to Vectors")
print("=" * 70)

chunk_texts = [chunk.page_content for chunk in chunks]
chunk_embeddings = model.encode(chunk_texts)

print(f"[SUCCESS] Embedded: {chunk_embeddings.shape[0]} chunks")
print(f"Vector dimensions: {chunk_embeddings.shape[1]}")
print(f"Model: all-MiniLM-L6-v2 (SentenceTransformer)")


# ============================================================
# STEP 4: STORE - Save Vectors in Qdrant Cloud
# ============================================================
print("\n" + "=" * 70)
print("STEP 4: STORE - Save Vectors in Qdrant Cloud")
print("=" * 70)

# Delete existing collection if it exists
try:
    client.delete_collection(collection_name="nlp_knowledge")
    print("[INFO] Deleted existing 'nlp_knowledge' collection")
except:
    print("[INFO] No existing collection to delete")

# Create new collection
client.create_collection(
    collection_name="nlp_knowledge",
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,      # 384 dimensions
        distance=Distance.COSINE,
    ),
)
print(f"[SUCCESS] Created new collection: 'nlp_knowledge' ({EMBEDDING_DIM} dimensions)")

# Build and upload points
points = []
for i, (text, embedding) in enumerate(zip(chunk_texts, chunk_embeddings)):
    points.append(
        PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload={
                "text": text,
                "source": "nlp_article.txt",
                "chunk_index": i,
            },
        )
    )

client.upsert(collection_name="nlp_knowledge", points=points)
print(f"[SUCCESS] Upserted {len(points)} points to Qdrant Cloud")

info = client.get_collection("nlp_knowledge")
print(f"[SUCCESS] Collection 'nlp_knowledge' now contains: {info.points_count} points")


# ============================================================
# PIPELINE COMPLETE!
# ============================================================
print("\n" + "=" * 70)
print("PIPELINE COMPLETE! You have built a full RAG knowledge base!")
print("=" * 70)
print("\nSummary of what you built:")
print(f"  [1] Loaded a text document from disk")
print(f"  [2] Split it into {len(chunks)} searchable chunks")
print(f"  [3] Converted each chunk into a {EMBEDDING_DIM}-dimensional vector (FREE)")
print(f"  [4] Stored everything in Qdrant Cloud with metadata")
print(f"  [5] Ready to search by meaning in the next cell")
print("\nThis is EXACTLY how production RAG systems work!")
print("\nNext steps:")
print("  - Visit your Qdrant Cloud dashboard to explore the collection")
print("  - Run the next cell to search by meaning")
print("=" * 70)

STEP 0: SETUP - Initialize Model & Database Connection


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 324.10it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[SUCCESS] Model loaded: all-MiniLM-L6-v2 (384 dimensions)
[SUCCESS] Connected to Qdrant Cloud

STEP 1: LOAD - Read Documents from Disk
[SUCCESS] Loaded: 1 document(s)
Preview: Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?
...

STEP 2: SPLIT - Break Documents into Chunks
[SUCCESS] Split into: 21 chunks
Chunk size: 500 characters
Chunk overlap: 50 characters
First chunk preview: Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural L...

STEP 3: EMBED - Convert Text to Vectors
[SUCCESS] Embedded: 21 chunks
Vector dimensions: 384
Model: all-MiniLM-L6-v2 (SentenceTransformer)

STEP 4: STORE - Save Vectors in Qdrant Cloud
[INFO] Deleted existing 'nlp_knowledge' collection
[SUCCESS] Created new collection: 'nlp_knowledge' (384 dimensions)
[SUCCESS] Upserted 21 points to Qdrant Cloud
[SUCCESS] Collection 'nlp_knowledge' now contains: 21 points

PIPELINE COMPLETE! You have built a full RAG knowledge base!

Summary o

In [ ]:
# ============================================================
# FULL RAG PIPELINE: Load -> Split -> Embed -> Store -> Search
# ============================================================
# EMBEDDING METHOD: OpenAI Embeddings (PAID, requires API key)
#
# This cell uses OpenAI embeddings (requires API key and costs money)
# For FREE local embeddings, use the previous cell instead
#
# Pipeline Steps:
#   0. Setup (OpenAI Client + Database)
#   1. Load (Read documents from disk)
#   2. Split (Break into chunks)
#   3. Embed (Convert to vectors with OpenAI API)
#   4. Store (Save in vector database)
#
# IMPORTANT: You need to set your OpenAI API key below!
# ============================================================

# ============================================================
# IMPORTS
# ============================================================
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from openai import OpenAI
import numpy as np
import os


# ============================================================
# STEP 0: SETUP - Initialize OpenAI Client & Database Connection
# ============================================================
print("=" * 70)
print("STEP 0: SETUP - Initialize OpenAI Client & Database Connection")
print("=" * 70)

# Set your OpenAI API key
# Get your API key from: https://platform.openai.com/api-keys
os.environ["OPENAI_API_KEY"] = ""  # Replace with your actual key
openai_client = OpenAI()

EMBEDDING_DIM = 1536  # OpenAI text-embedding-3-small output size
print("[SUCCESS] OpenAI client initialized (1536 dimensions)")
print("[INFO] Model: text-embedding-3-small")

# Connect to Qdrant Cloud
QDRANT_URL = ""
QDRANT_API_KEY = ""
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
print("[SUCCESS] Connected to Qdrant Cloud")


# ============================================================
# STEP 1: LOAD - Read Documents from Disk
# ============================================================
print("\n" + "=" * 70)
print("STEP 1: LOAD - Read Documents from Disk")
print("=" * 70)

loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
raw_documents = loader.load()

print(f"[SUCCESS] Loaded: {len(raw_documents)} document(s)")
print(f"Preview: {raw_documents[0].page_content[:100]}...")


# ============================================================
# STEP 2: SPLIT - Break Documents into Chunks
# ============================================================
print("\n" + "=" * 70)
print("STEP 2: SPLIT - Break Documents into Chunks")
print("=" * 70)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = splitter.split_documents(raw_documents)

print(f"[SUCCESS] Split into: {len(chunks)} chunks")
print(f"Chunk size: 500 characters")
print(f"Chunk overlap: 50 characters")
print(f"First chunk preview: {chunks[0].page_content[:80]}...")


# ============================================================
# STEP 3: EMBED - Convert Text to Vectors (OpenAI API)
# ============================================================
print("\n" + "=" * 70)
print("STEP 3: EMBED - Convert Text to Vectors")
print("=" * 70)

chunk_texts = [chunk.page_content for chunk in chunks]

# Function to get OpenAI embeddings
def get_openai_embedding(text):
    """Get embedding from OpenAI API."""
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",  # or use "text-embedding-ada-002"
        input=text
    )
    return response.data[0].embedding

# Embed all chunks using OpenAI API
chunk_embeddings = []
print(f"[INFO] Embedding {len(chunk_texts)} chunks using OpenAI API...")

for i, text in enumerate(chunk_texts):
    embedding = get_openai_embedding(text)
    chunk_embeddings.append(embedding)
    
    # Progress update every 5 chunks
    if (i + 1) % 5 == 0 or (i + 1) == len(chunk_texts):
        print(f"[PROGRESS] Embedded {i + 1}/{len(chunk_texts)} chunks")

# Convert to numpy array for consistency
chunk_embeddings = np.array(chunk_embeddings)

print(f"[SUCCESS] Embedded: {chunk_embeddings.shape[0]} chunks")
print(f"Vector dimensions: {chunk_embeddings.shape[1]}")
print(f"Model: OpenAI text-embedding-3-small")


# ============================================================
# STEP 4: STORE - Save Vectors in Qdrant Cloud
# ============================================================
print("\n" + "=" * 70)
print("STEP 4: STORE - Save Vectors in Qdrant Cloud")
print("=" * 70)

# Delete existing collection if it exists
try:
    client.delete_collection(collection_name="nlp_knowledge_openai")
    print("[INFO] Deleted existing 'nlp_knowledge_openai' collection")
except:
    print("[INFO] No existing collection to delete")

# Create new collection with 1536 dimensions (OpenAI size)
client.create_collection(
    collection_name="nlp_knowledge_openai",
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,      # 1536 dimensions for OpenAI
        distance=Distance.COSINE,
    ),
)
print(f"[SUCCESS] Created new collection: 'nlp_knowledge_openai' ({EMBEDDING_DIM} dimensions)")

# Build and upload points
points = []
for i, (text, embedding) in enumerate(zip(chunk_texts, chunk_embeddings)):
    points.append(
        PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload={
                "text": text,
                "source": "nlp_article.txt",
                "chunk_index": i,
            },
        )
    )

client.upsert(collection_name="nlp_knowledge_openai", points=points)
print(f"[SUCCESS] Upserted {len(points)} points to Qdrant Cloud")

info = client.get_collection("nlp_knowledge_openai")
print(f"[SUCCESS] Collection 'nlp_knowledge_openai' now contains: {info.points_count} points")


# ============================================================
# PIPELINE COMPLETE!
# ============================================================
print("\n" + "=" * 70)
print("PIPELINE COMPLETE! You have built a full RAG knowledge base!")
print("=" * 70)
print("\nSummary of what you built:")
print(f"  [1] Loaded a text document from disk")
print(f"  [2] Split it into {len(chunks)} searchable chunks")
print(f"  [3] Converted each chunk into a {EMBEDDING_DIM}-dimensional vector (OpenAI)")
print(f"  [4] Stored everything in Qdrant Cloud with metadata")
print(f"  [5] Ready to search by meaning in the next cell")
print("\nThis is EXACTLY how production RAG systems work!")
print("\nComparison with SentenceTransformer:")
print("  - OpenAI: Higher quality, 1536 dims, requires API key, costs money")
print("  - SentenceTransformer: Free, 384 dims, runs locally, good for learning")
print("\nNext steps:")
print("  - Visit your Qdrant Cloud dashboard to explore the collection")
print("  - Compare search quality between OpenAI and SentenceTransformer embeddings")
print("=" * 70)

---

## 9. Mini Challenges

### Challenge 1: The Genre Explorer
Add 5 more movies to the `"movies"` collection (upsert them with IDs 11-15).  
Include at least 2 different genres. Then search for  
`"a funny comedy about friends"` and see which movies come up.

### Challenge 2: Advanced Filtering
Using the movies collection, write a search that finds movies about  
"heroes and villains" but filters for **only movies released after 2000**.  
Hint: Use `Range` from `qdrant_client.models` with `gt=2000` for the year field.

```python
from qdrant_client.models import Range
FieldCondition(key="year", range=Range(gt=2000))
```

### Challenge 3: Your Own Knowledge Base
Take any text you like (a Wikipedia article, a blog post, or your own notes).  
Build the full pipeline:
1. Split it into chunks with `RecursiveCharacterTextSplitter`
2. Embed all chunks with `model.encode()`
3. Store them in a new Qdrant collection
4. Write 3 search queries and test them

In [ ]:
# ============================================================
# Challenge 1: The Genre Explorer
# ============================================================
# Add 5 more movies (IDs 11-15) with different genres
# Then search for "a funny comedy about friends"

# Your code here:


In [ ]:
# ============================================================
# Challenge 2: Advanced Filtering
# ============================================================
# Search for "heroes and villains" but only movies after 2000
# Hint: Use Range(gt=2000) for the year field

# Your code here:


In [ ]:
# ============================================================
# Challenge 3: Your Own Knowledge Base
# ============================================================
# 1. Choose a text (any topic you like!)
# 2. Split -> Embed -> Store in Qdrant -> Search
# Use the full pipeline from Section 8 as your template

# Your code here:


---

## 10. Quick Reference: Vector Store Cheat Sheet

### Qdrant Setup Patterns

```python
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Qdrant Cloud (recommended â€” free tier available!)
QDRANT_URL = ""
QDRANT_API_KEY = ""
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# In-memory (quick testing only â€” data disappears on restart)
client = QdrantClient(":memory:")

# Local file storage (prototyping)
client = QdrantClient(path="./qdrant_data")
```

### Core Operations

```python
# CREATE collection
client.create_collection(
    collection_name="my_collection",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

# UPSERT points
client.upsert(
    collection_name="my_collection",
    points=[PointStruct(id=1, vector=[...], payload={"key": "value"})],
)

# CREATE PAYLOAD INDEX (required for Qdrant Cloud before filtering!)
client.create_payload_index(
    collection_name="my_collection",
    field_name="category",
    field_schema="keyword",  # "keyword" for text, "integer" for numbers
)

# SEARCH (similarity search with pre-computed vectors)
results = client.query_points(
    collection_name="my_collection",
    query=[...],  # Your embedding vector as a list
    limit=5,
).points

# SEARCH with filter
from qdrant_client.models import Filter, FieldCondition, MatchValue
results = client.query_points(
    collection_name="my_collection",
    query=[...],
    query_filter=Filter(
        must=[FieldCondition(key="category", match=MatchValue(value="tech"))]
    ),
    limit=5,
).points

# DELETE by IDs
from qdrant_client.models import PointIdsList
client.delete(
    collection_name="my_collection",
    points_selector=PointIdsList(points=[1, 2, 3]),
)
```

### Important Notes

- **Payload indexes are required for Qdrant Cloud!** Create indexes for any field you want to filter on
- Field schemas: `"keyword"` (text), `"integer"` (numbers), `"float"` (decimals), `"bool"` (boolean)
- In-memory mode doesn't require indexes, but cloud/server mode does

### The Golden Rule

> **Always embed your query with the SAME model you used to embed your documents.**  
> Mixing models (e.g., embedding docs with MiniLM but querying with OpenAI)  
> will give meaningless results because the vector spaces are different.

---

## 11. Key Takeaways

1. **Regular databases can't handle vector search** â€” brute force is O(n) and too slow at scale
2. **Vector databases** are specialized for storing and searching embeddings efficiently
3. **Qdrant** = open-source, Rust-based, free self-hosting, great for full control
4. **Pinecone** = fully managed, zero setup, great for quick deployment
5. **Four core operations:** upsert (add), query (search), filter (narrow), delete (remove)
6. **HNSW algorithm** = multi-layer graph that makes search O(log n) â€” milliseconds even at billions!
7. **The RAG pipeline so far:**

```
Load (L5) --> Split (L6) --> Embed (L7) --> Store & Search (L8) --> Retrieve + Answer (coming next!)
```

### Next Lecture

**Lecture 9: Making Retrieval Smarter with MMR** â€” You can find similar  
documents, but what if the top 5 results all say the same thing?  
MMR (Maximum Marginal Relevance) adds diversity to your search results!

---

*Hope to Skill â€” Building the future, one skill at a time.*

---

## Appendix: PEP 8 Style Rules Used in This Notebook

All code in this notebook follows Python's PEP 8 style guide.  
Here are the rules applied, with examples from the code above.

### Naming Conventions

| Rule | Convention | Example from This Notebook |
|------|-----------|---------------------------|
| Variables & functions | `snake_case` | `query_embedding`, `chunk_texts`, `rag_search()` |
| Constants | `UPPER_CASE` | None used (configurations are variable) |
| Classes | `PascalCase` | `QdrantClient`, `PointStruct`, `SentenceTransformer` (from libraries) |

### Import Rules

| Rule ID | Rule | Example |
|---------|------|---------|  
| E401 | One import per line | `from qdrant_client import QdrantClient` |
| E402 | Imports at the top of each section | All imports at cell top |
| â€” | Group: stdlib â†’ third-party â†’ local | `time` â†’ `numpy` â†’ `qdrant_client` |

### Whitespace & Formatting

| Rule ID | Rule | Example |
|---------|------|---------|  
| E225 | Spaces around operators | `rank + 1`, `score > 0` |
| E231 | Space after commas | `PointStruct(id=1, vector=[...])` |
| E265 | Block comments start with `# ` | `# Embed the query using the same model` |
| E303 | Two blank lines before function defs | See `rag_search()` definition |
| E501 | Max line length of 79 characters | Long strings use wrapping |

### Best Practices Used

| Practice | Why | Example |
|----------|-----|---------|  
| f-strings | Clean string formatting | `f"Score: {result.score:.4f}"` |
| `enumerate()` | Index + value in loops | `for i, movie in enumerate(movies)` |
| `zip()` | Pair two lists | `for text, emb in zip(chunk_texts, chunk_embeddings)` |
| List comprehensions | Concise list building | `[chunk.page_content for chunk in chunks]` |
| `:.4f` formatting | Control decimal places | `result.score:.4f` |
| Docstrings | Explain function purpose | `rag_search()` has a docstring |
| Descriptive names | Code reads like English | `query_embedding` not `qe` |
| `.tolist()` conversion | Qdrant needs plain lists | `embedding.tolist()` |

### Quick PEP 8 Cheat Sheet

```python
# GOOD (PEP 8 compliant)
query_embedding = model.encode("search query")
results = client.search(
    collection_name="movies",
    query_vector=query_embedding.tolist(),
    limit=3,
)
for rank, result in enumerate(results):
    print(f"#{rank + 1}: {result.payload['title']}")

# BAD (violates PEP 8)
qe = model.encode("search query")                          # unclear name
res = client.search(collection_name="movies",query_vector=qe.tolist(),limit=3)  # no spaces
for i,r in enumerate(res):                                 # single-letter vars
    print("#"+str(i+1)+": "+r.payload['title'])           # string concat, no f-string
```